In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

#  Two ways to do summarization in modern langchain

1. `Stuff document` - send all docs together to llm and ask to summarize (modern llm models with high context window can handle but not   ideal method) 

![Map reduce ](images/1_zrqpzBNG0AJfU2Qvo6hpuQ.webp)

2. `Map reduce`- 

![Map reduce ](images/map-reduce-summary.max-1900x1900.png)

### Stuff document

In [2]:
from langchain_ollama import ChatOllama

/Users/adityakumar/Developer/Code/genai/langchain_projects/text_summarization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [68]:
#test


llm=ChatOllama(model='gemma4',num_ctx=50000)
llm.invoke('hello')

AIMessage(content='Hello! 👋 How are you doing today? Is there anything specific I can help you with?', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-06-17T11:48:59.419684Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24353691792, 'load_duration': 4798425542, 'prompt_eval_count': 11, 'prompt_eval_duration': 435535000, 'eval_count': 329, 'eval_duration': 19118410000, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019ed569-6679-7be1-a354-9fc89f455d56-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 329, 'total_tokens': 340})

In [60]:
generic_prompt = """
You are an expert multilingual document analyst.

Analyze the document and produce:

1. A comprehensive Markdown summary.
2. Preserve every important section and heading.
3. Capture:
   - Main ideas
   - Definitions
   - Key concepts
   - Important examples
   - Numerical data
   - Conclusions
   - Action items (if any)
4. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.
5. Maintain the logical flow of the original document.
6. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.
7. After the summary, provide a high-quality translation in {language}.

Output:

# Detailed Summary
...

# Translation ({language})
...
"""

In [191]:
from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import SystemMessage,HumanMessage

loader= PyPDFLoader('source_docs/bert.pdf')
docs=loader.load()


summarizer_prompt=ChatPromptTemplate.from_messages([
    ('system',generic_prompt) ,
    ('human','document : {text}')
      
        ])


summarizer_prompt

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='\nYou are an expert multilingual document analyst.\n\nAnalyze the document and produce:\n\n1. A comprehensive Markdown summary.\n2. Preserve every important section and heading.\n3. Capture:\n   - Main ideas\n   - Definitions\n   - Key concepts\n   - Important examples\n   - Numerical data\n   - Conclusions\n   - Action items (if any)\n4. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.\n5. Maintain the logical flow of the original document.\n6. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.\n7. After the summary, provide a high-quality translation in {language}.\n\nOutput:\n\n# Detailed Summary\n...\n\n# Translation ({language})\n...\n'), additional_kwarg

In [192]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'source_docs/bert.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='BERT: Pre-training of Deep Bidirectional Transformers for\nLanguage Understanding\nJacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova\nGoogle AI Language\n{jacobdevlin,mingweichang,kentonl,kristout}@google.com\nAbstract\nWe introduce a new language representa-\ntion model called BERT, which stands for\nBidirectional Encoder Representations from\nTransformers. Unlike recent language repre-\nsentation models (Peters et al., 2018a; Rad-\nford et al., 2018), BERT is designed to pre-\ntrain deep bidirectional representations from\

In [193]:
summarizer_prompt.input_variables

['language', 'text']

In [194]:
summarizer_chain=summarizer_prompt|llm

In [195]:
text=" ".join([doc.page_content for doc in docs])

In [196]:
from langchain_core.messages.utils import count_tokens_approximately

In [198]:
from langchain_core.messages.utils import count_tokens_approximately


print('token count of whole doc :' ,count_tokens_approximately(text))



token count of whole doc : 19500
token count after limiting doc to max token : 254225


In [185]:
summarizer_prompt.invoke({'text':" ".join([doc.page_content for doc in docs]),'language':'hindi'})

ChatPromptValue(messages=[SystemMessage(content='\nYou are an expert multilingual document analyst.\n\nAnalyze the document and produce:\n\n1. A comprehensive Markdown summary.\n2. Preserve every important section and heading.\n3. Capture:\n   - Main ideas\n   - Definitions\n   - Key concepts\n   - Important examples\n   - Numerical data\n   - Conclusions\n   - Action items (if any)\n4. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.\n5. Maintain the logical flow of the original document.\n6. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.\n7. After the summary, provide a high-quality translation in hindi.\n\nOutput:\n\n# Detailed Summary\n...\n\n# Translation (hindi)\n...\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='document : BERT: Pre-training of Deep Bidirectional Transformers for\nLanguage Understanding\nJacob Devlin Ming-Wei Chang Kenton Lee Kristina To

In [186]:
summarizer_chain.invoke({'text':text,'language':'hindi'})

KeyboardInterrupt: 

`Ǹote : Stuff chain is not good with models with less context window.`

In [65]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='llama-3.3-70b-versatile')

In [66]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x11518f650>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x115177290>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [120]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model='qwen3.5',temperature=0.0,reasoning=False, num_ctx=60768)

In [121]:
from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate


In [122]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader= PyPDFLoader('source_docs/bert.pdf')
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splited=splitter.split_documents(docs)


In [123]:
chunk_summary_generic_prompt = """
You are an expert information extraction and summarization system.

Extract and compress the information in this chunk.

Preserve:
- concepts
- facts
- numbers
- formulas
- terminology

Remove repetition.
Do not invent information.
Keep maximum information in minimum words.


"""


In [124]:
chunk_summary_prompt=ChatPromptTemplate.from_messages(
    [
        ('system',chunk_summary_generic_prompt),
        
        ('human','{text}')
    ]
)


In [125]:
chunk_summary_prompt

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nYou are an expert information extraction and summarization system.\n\nExtract and compress the information in this chunk.\n\nPreserve:\n- concepts\n- facts\n- numbers\n- formulas\n- terminology\n\nRemove repetition.\nDo not invent information.\nKeep maximum information in minimum words.\n\n\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [126]:
chunk_summ_chain=chunk_summary_prompt|llm

In [127]:
splited

[Document(metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-05-28T00:07:51+00:00', 'author': '', 'keywords': '', 'moddate': '2019-05-28T00:07:51+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'source_docs/bert.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='BERT: Pre-training of Deep Bidirectional Transformers for\nLanguage Understanding\nJacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova\nGoogle AI Language\n{jacobdevlin,mingweichang,kentonl,kristout}@google.com\nAbstract\nWe introduce a new language representa-\ntion model called BERT, which stands for\nBidirectional Encoder Representations from\nTransformers. Unlike recent language repre-\nsentation models (Peters et al., 2018a; Rad-\nford et al., 2018), BERT is designed to pre-\ntrain deep bidirectional representations from\

In [128]:
len(splited)

83

In [46]:

text = "\n".join(chunk.page_content for chunk in splited)

In [47]:

print("Characters:", len(text))
print("Words:", len(text.split()))

Characters: 76216
Words: 12060


In [48]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

tokens = len(enc.encode(text))

print(tokens)


20919


In [49]:
all_chunks = []

for i, chunk in enumerate(splited, start=1):
    try:
        result = chunk_summ_chain.invoke(
            {"text": chunk.page_content}
        )

        all_chunks.append(result.content)
        print(result.content)

        print(f"Chunk {i} done")

    except Exception as e:
        print(f"Chunk {i} failed:", e)
       

**BERT (Bidirectional Encoder Representations from Transformers)**: A deep language representation model by Google AI Language that pre-trains bidirectional representations on unlabeled text using left/right context conditioning across all layers, enabling fine-tuning with one output layer for state-of-the-art performance in tasks like question answering and inference without major architectural changes.
Chunk 1 done
BERT achieves state-of-the-art results on eleven NLP tasks via language model pre-training without substantial architectural changes. It improves GLUE (80.5%, +7.7%), MultiNLI (86.7%, +4.6%), SQuAD v1.1 F1 (93.2, +1.5), and SQuAD v2.0 F1 (83.1, +5.1). Pre-training enhances sentence-level tasks like natural language inference and paraphrasing to predict inter-sentence relationships.
Chunk 2 done
Natural language inference (Bowman et al., 2015; Williams et al., 2018) and paraphrasing (Dolan & Brockett, 2005) analyze sentences holistically to predict relationships. Token-leve

In [58]:
#through async batch

inputs = [
    {"text": chunk.page_content}
    for chunk in splited
]

result_with_batch=await chunk_summ_chain.abatch(inputs,  config={"max_concurrency": 4})

In [67]:
result_with_batch

[AIMessage(content='**BERT (Bidirectional Encoder Representations from Transformers)**: A deep language representation model by Google AI Language that pre-trains bidirectional representations on unlabeled text using left/right context conditioning across all layers, enabling fine-tuning with one output layer for state-of-the-art performance in tasks like question answering and inference without major architectural changes.', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-06-17T16:37:41.333425Z', 'done': True, 'done_reason': 'stop', 'total_duration': 38881491208, 'load_duration': 4402633625, 'prompt_eval_count': 315, 'prompt_eval_duration': 1935583000, 'eval_count': 68, 'eval_duration': 3885753000, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019ed671-7d67-7931-b54c-50c3d09e90af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 68, 'total_tokens': 383}),
 AIMessage(con

In [129]:
generic_prompt = """
You are an expert multilingual document analyst.

Analyze the document and produce:

1. A comprehensive Markdown summary.
2. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.
3. Maintain the logical flow of the original document.
4. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.
5. After the summary, provide a high-quality translation in {language}.

Output:

# Detailed Summary
...

# Translation 
...
"""

In [130]:
summarizer_prompt=ChatPromptTemplate.from_messages([
    ('system',generic_prompt) ,
    ('human','document : {text}')
      
        ])


In [131]:
reducer_chain=summarizer_prompt|llm

In [132]:
reducer_chain

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='\nYou are an expert multilingual document analyst.\n\nAnalyze the document and produce:\n\n1. A comprehensive Markdown summary.\n2. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.\n3. Maintain the logical flow of the original document.\n4. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.\n5. After the summary, provide a high-quality translation in {language}.\n\nOutput:\n\n# Detailed Summary\n...\n\n# Translation \n...\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='document : {text}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langch

In [133]:
chunked_sum='\n'.join([chunk for chunk in all_chunks])

In [134]:
summarizer_prompt.invoke({'text':chunked_sum,'language':'hindi'})

ChatPromptValue(messages=[SystemMessage(content='\nYou are an expert multilingual document analyst.\n\nAnalyze the document and produce:\n\n1. A comprehensive Markdown summary.\n2. Do not shorten excessively. Aim to retain 80 to 90 percent of the informational value while removing repetition.\n3. Maintain the logical flow of the original document.\n4. Use nested Markdown headings, bullet points, numbered lists, and tables where useful.\n5. After the summary, provide a high-quality translation in hindi.\n\nOutput:\n\n# Detailed Summary\n...\n\n# Translation \n...\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='document : **BERT (Bidirectional Encoder Representations from Transformers)**: A deep language representation model by Google AI Language that pre-trains bidirectional representations on unlabeled text using left/right context conditioning across all layers, enabling fine-tuning with one output layer for state-of-the-art performance in tasks like question an

In [135]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

tokens = len(enc.encode(chunked_sum))

print(tokens)


14616


In [136]:
final_result_summary=reducer_chain.invoke({'text':chunked_sum,'language':'hindi'})
final_result_summary

AIMessage(content='# Detailed Summary\n\n## **BERT: Bidirectional Encoder Representations from Transformers**\n\n### **1. Overview & Core Innovation**\n*   **Definition:** BERT is a deep language representation model developed by Google AI Language that pre-trains bidirectional representations on unlabeled text using left/right context conditioning across all layers.\n*   **Key Advantage:** It enables fine-tuning with only one output layer, achieving state-of-the-art (SOTA) performance in tasks like question answering and inference without major architectural changes.\n*   **Performance Impact:** BERT achieves SOTA results on eleven NLP tasks via language model pre-training:\n    *   Improves GLUE benchmark by **+7.7%** (80.5%).\n    *   Improves MultiNLI accuracy to 86.7% (+4.6%).\n    *   Achieves F1 scores of **93.2** on SQuAD v1.1 and **83.1** on SQuAD v2.0.\n\n### **2. Architectural Comparison: BERT vs. GPT/ELMo**\n*   **Bidirectionality:** Unlike OpenAI\'s GPT (which uses unidire

In [142]:
print(final_result_summary.content)

# Detailed Summary

## **BERT: Bidirectional Encoder Representations from Transformers**

### **1. Overview & Core Innovation**
*   **Definition:** BERT is a deep language representation model developed by Google AI Language that pre-trains bidirectional representations on unlabeled text using left/right context conditioning across all layers.
*   **Key Advantage:** It enables fine-tuning with only one output layer, achieving state-of-the-art (SOTA) performance in tasks like question answering and inference without major architectural changes.
*   **Performance Impact:** BERT achieves SOTA results on eleven NLP tasks via language model pre-training:
    *   Improves GLUE benchmark by **+7.7%** (80.5%).
    *   Improves MultiNLI accuracy to 86.7% (+4.6%).
    *   Achieves F1 scores of **93.2** on SQuAD v1.1 and **83.1** on SQuAD v2.0.

### **2. Architectural Comparison: BERT vs. GPT/ELMo**
*   **Bidirectionality:** Unlike OpenAI's GPT (which uses unidirectional left-to-right Transformer